# Task 2.1: Estrategias de generación

## Imports y reproducibilidad

In [1]:
import math
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

device: mps


## Modelo

In [2]:
import math
import time

import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps) * gamma + beta


def make_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.WQ = nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.WK = nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.WV = nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.WO = nn.Parameter(torch.randn(d_model, d_model) * 0.02)

        self.gamma1 = nn.Parameter(torch.ones(d_model))
        self.beta1 = nn.Parameter(torch.zeros(d_model))

        self.W1 = nn.Parameter(torch.randn(d_model, d_ff) * 0.02)
        self.bias1 = nn.Parameter(torch.zeros(d_ff))
        self.W2 = nn.Parameter(torch.randn(d_ff, d_model) * 0.02)
        self.bias2 = nn.Parameter(torch.zeros(d_model))

        self.gamma2 = nn.Parameter(torch.ones(d_model))
        self.beta2 = nn.Parameter(torch.zeros(d_model))

    def masked_self_attention(self, x, causal_mask):
        B, T, C = x.shape
        Q = (x @ self.WQ).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = (x @ self.WK).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = (x @ self.WV).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, n_heads, T, T)
        scores = scores.masked_fill(causal_mask[:T, :T], float("-inf"))
        attn_w = F.softmax(scores, dim=-1)

        out = (attn_w @ V).transpose(1, 2).contiguous().view(B, T, C)
        return out @ self.WO

    def feed_forward(self, x):
        h = F.relu(x @ self.W1 + self.bias1)
        return h @ self.W2 + self.bias2

    def forward(self, x, causal_mask):
        # pre-LayerNorm (estilo GPT-2): estabiliza el entrenamiento al apilar capas
        x = x + self.masked_self_attention(layer_norm(x, self.gamma1, self.beta1), causal_mask)
        x = x + self.feed_forward(layer_norm(x, self.gamma2, self.beta2))
        return x


class MiniGPT(nn.Module):
    def __init__(self, V, d_model, n_heads, n_layers, block_size):
        super().__init__()
        self.E = nn.Parameter(torch.randn(V, d_model) * 0.02)
        self.register_buffer("PE", make_positional_encoding(block_size, d_model))
        self.register_buffer("causal_mask", torch.triu(torch.ones(block_size, block_size), diagonal=1).bool())

        self.blocks = nn.ModuleList([Block(d_model, n_heads, 4 * d_model) for _ in range(n_layers)])

        self.gamma_f = nn.Parameter(torch.ones(d_model))
        self.beta_f = nn.Parameter(torch.zeros(d_model))
        self.Wlm = nn.Parameter(torch.randn(d_model, V) * 0.02)
        self.blm = nn.Parameter(torch.zeros(V))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.E[idx] + self.PE[:T]
        for block in self.blocks:
            x = block(x, self.causal_mask)
        x = layer_norm(x, self.gamma_f, self.beta_f)
        logits = x @ self.Wlm + self.blm  # (B, T, V)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

## Carga del checkpoint

In [3]:
CKPT_PATH = os.path.join("checkpoints", "mini_gpt.pt")
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"No existe {CKPT_PATH}: ejecute task1.ipynb primero para entrenar el modelo.")

ckpt = torch.load(CKPT_PATH, map_location="cpu")
stoi, itos, config = ckpt["stoi"], ckpt["itos"], ckpt["config"]
block_size = config["block_size"]
vocab_size = len(stoi)

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

model = MiniGPT(vocab_size, **config).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

val_loss = ckpt["history"]["val"][-1]
print("config:", config, "| vocab_size:", vocab_size)
print(f"val loss: {val_loss:.4f} | perplejidad: {math.exp(val_loss):.4f}")

config:

 {'block_size': 128, 'd_model': 128, 'n_heads': 4, 'n_layers': 3} | vocab_size: 65
val loss: 1.9682 | perplejidad: 7.1580


## Estrategias de muestreo

In [4]:
def greedy(logits):
    return int(torch.argmax(logits))


def top_k(logits, k, tau=1.0):
    values, indices = torch.topk(logits / tau, k)
    probs = F.softmax(values, dim=-1)
    return int(indices[torch.multinomial(probs, num_samples=1)])


def top_p(logits, p, tau=1.0):
    probs = F.softmax(logits / tau, dim=-1)
    sorted_probs, indices = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    # conjunto mínimo cuya probabilidad acumulada alcanza p (siempre incluye el más probable)
    keep = (cumulative - sorted_probs) < p
    kept_probs = sorted_probs[keep] / sorted_probs[keep].sum()
    return int(indices[keep][torch.multinomial(kept_probs, num_samples=1)])

## Función generate

In [5]:
STRATEGIES = {"greedy": greedy, "top-k": top_k, "top-p": top_p}


@torch.no_grad()
def generate(prompt, max_new_tokens, strategy, **kwargs):
    if strategy not in STRATEGIES:
        raise ValueError(f"Estrategia desconocida: {strategy!r}. Opciones: {list(STRATEGIES)}")
    pick = STRATEGIES[strategy]

    model.eval()
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        context = torch.tensor([ids[-block_size:]], dtype=torch.long, device=device)
        logits, _ = model(context)
        ids.append(pick(logits[0, -1].cpu(), **kwargs))
    return decode(ids)

## Configuraciones

In [6]:
PROMPT = "ROMEO:"
N_SAMPLES = 5
MAX_NEW_TOKENS = 200

CONFIGS = {
    "A": ("greedy", {}),
    "B": ("top-k", dict(k=5, tau=0.7)),
    "C": ("top-k", dict(k=50, tau=1.0)),
    "D": ("top-p", dict(p=0.70, tau=0.7)),
    "E": ("top-p", dict(p=0.95, tau=1.0)),
}

## Muestras

In [7]:
torch.manual_seed(SEED)
samples = {}
for name, (strategy, kwargs) in CONFIGS.items():
    samples[name] = [generate(PROMPT, MAX_NEW_TOKENS, strategy, **kwargs) for _ in range(N_SAMPLES)]
    print(f"=== {name} · {strategy} · {kwargs} ===")
    for i, text in enumerate(samples[name], 1):
        print(f"--- muestra {i} ---")
        print(text)
        print()

=== A · greedy · {} ===
--- muestra 1 ---
ROMEO:
I what the the the the the the the the the the the the the the
To the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the t

--- muestra 2 ---
ROMEO:
I what the the the the the the the the the the the the the the
To the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the t

--- muestra 3 ---
ROMEO:
I what the the the the the the the the the the the the the the
To the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the t

--- muestra 4 ---
ROMEO:
I what the the the the the the the the the the the the the the
To the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the t

--- muestra 5 ---
ROMEO:
I what the the the the the the the the the the 

=== B · top-k · {'k': 5, 'tau': 0.7} ===
--- muestra 1 ---
ROMEO:
What thou mis to were stalle would sernones
In hear here make will will will me with hencand.

LOREO:
I were have a the stoo and the with share me,
Tho hear hond tree thou with worse to see
The the m

--- muestra 2 ---
ROMEO:
I'll to woor that shall the we were make to thing
The hould to him, the should the thou sir a she
Whath will tay alle sir the alt souss a sin.

LAUSES:
The me that stour on the and sto the stay.

MUL

--- muestra 3 ---
ROMEO:
Ay that the soun soul sour the a shume
Whe here shous thing hance an his me the hone
And stise to me stongs angert and streath
The and with whou see to angers, this with with and worse.

Serive, then

--- muestra 4 ---
ROMEO:
What make to him the to maress,
We may the have thou wart to the thich was shand
Than a stall the that there are this then mise
To shane strere ont thou with and me my tay.

She work, seet and to thi

--- muestra 5 ---
ROMEO:
I with marry the that would an

=== C · top-k · {'k': 50, 'tau': 1.0} ===
--- muestra 1 ---
ROMEO:
I'll of is hear dentle made,--But thour,
Anvaice lain; and
Then upememen loody tecown theamight
And coutin's am here shicky crawn notiomely
An chorat as mery I counsled it aganst.
Tha mangur sterer; 

--- muestra 2 ---
ROMEO:
Thr-me the hawshord no is the bepriegatiome.

ENEO:
Nome, t to ofath anter, a shy litgornt, For dord?

HERCUCERY:
Doo mosthan ploing cor' thou RGjo:
That to's sirser, may will ancous; wheat that
Mare

--- muestra 3 ---
ROMEO:
Sen it prilad contelt to doh that timinis.

MERUTCKEs--revers
Thance bothere hour to Trist not wele,
Hand pance almenout how and them es&ashtlerss,
Prilistior
Thegu it thid meands of, cleast same pea

--- muestra 4 ---
ROMEO:
The cousey day'd the my beg heinge?

MAMORDTEO:
I tnemowerd, At mariows:-houary, 'sin?

MEENEO:

C'et morer:
Is fay my food sive, but see and you I nouve;

Suir a spution ort good lave; but ent dove 

--- muestra 5 ---
ROMEO:
Mith on couse may mich will,


=== D · top-p · {'p': 0.7, 'tau': 0.7} ===
--- muestra 1 ---
ROMEO:
That some have be to the with strown,
What the are the a coust here the faintent
The that with him me that in for in that a me.

MAULINCE:
I with me be stay a him seat on the me a mont were,
And the 

--- muestra 2 ---
ROMEO:
What be word beaing the brother him the with
Whe have me make of soul man will and the brow
And the the will with a do in on the seat with the my mand.

Fir will the in be the did with word that in m

--- muestra 3 ---
ROMEO:
I she that the shall be stay sour that my thee
are his the to his that with and his that my be hand
I more worse that the causter hear hears
Whe bean on of the that stand and with thee wear of a than

--- muestra 4 ---
ROMEO:
That be it that with may hath the the sour the to the and a the
To mane of the with the with with be more,
And staing one an the in the the be a the be then the with but him in and a have
That he hav

--- muestra 5 ---
ROMEO:
Whou me me my that shall sou

=== E · top-p · {'p': 0.95, 'tau': 1.0} ===
--- muestra 1 ---
ROMEO:
Fare whould benign: thou paur toe!

CLANET:
For his exing mut the hord reforectin.
Fill flathers on out nemast shey dim you it our shis a sar,
I novent not acains int of you do greatuo,
Hell mandance

--- muestra 2 ---
ROMEO:
I sexplie upoumpast freant a farteron;
That in hould end flowe beay in no gemmimin.

MUKENTY REO:
Than blingens, that that him lamithed I feem and
Ertricht knour o'll the no anse of as it meand
If ab

--- muestra 3 ---
ROMEO:
Goright kidin. Bonoly! for cars winksing in to frice;
Tindely fim on of lane with like bage
To bely you sigh mysalst; that by diy.

PALEO:
By raccatar, now.

RIS:
Poir, my good weart minsen cont agas

--- muestra 4 ---
ROMEO:
A be comblint lave havind me, where feep.
'Th let rabiant his embuctie trang tissul
LOM:
Fay worster macth sill to ancers may alrear
graty's hig hearen brainst coin o' brownoss
I her than sure to upp

--- muestra 5 ---
ROMEO:
Pew sway matt didio.

QUEEN

## Verificación

In [8]:
for name, texts in samples.items():
    assert len(texts) == N_SAMPLES
    for text in texts:
        assert text.startswith(PROMPT)
        assert len(text) == len(PROMPT) + MAX_NEW_TOKENS

assert len(set(samples["A"])) == 1
assert generate(PROMPT, 50, "top-k", k=1, tau=1.0) == generate(PROMPT, 50, "greedy")
print("OK")

OK


# Task 2.2: Análisis de las muestras

## Análisis

In [9]:
import re
from collections import defaultdict


@torch.no_grad()
def next_char_logits(text):
    """Logits del siguiente carácter en cada posición de `text` (contexto deslizante de block_size)."""
    ids = encode(text)
    rows = []
    for j in range(len(ids) - 1):
        context = torch.tensor([ids[max(0, j + 1 - block_size) : j + 1]], dtype=torch.long, device=device)
        logits, _ = model(context)
        rows.append(logits[0, -1].cpu())
    return torch.stack(rows)


def nucleus_size(probs, p):
    sorted_probs, _ = torch.sort(probs, descending=True, dim=-1)
    return ((sorted_probs.cumsum(-1) - sorted_probs) < p).sum(-1)


def show(text):
    """Texto legible: los saltos de línea se muestran como [salto]."""
    return text.replace("\n", "[salto]")


def show_char(c):
    return {"\n": "[salto]", " ": "[espacio]"}.get(c, c)


START = len(PROMPT) - 1  # primera posición cuya predicción es un carácter generado
logits_cache = {name: [next_char_logits(t) for t in texts] for name, texts in samples.items()}

### Greedy (A)

In [10]:
text_a = samples["A"][0]
probs_a = F.softmax(logits_cache["A"][0], dim=-1)
for j in range(START, START + 18):
    top = torch.topk(probs_a[j], 3)
    options = ", ".join(f"{show_char(itos[int(i)])} {v:.2f}" for v, i in zip(top.values, top.indices))
    context = show(text_a[max(0, j - 10) : j + 1])
    print(f"{context:>22}  elige {show_char(text_a[j + 1]):9}  opciones: {options}")

                ROMEO:  elige [salto]    opciones: [salto] 0.99, [espacio] 0.00, . 0.00
         ROMEO:[salto]  elige I          opciones: I 0.18, T 0.14, W 0.10
        ROMEO:[salto]I  elige [espacio]  opciones: [espacio] 0.49, t 0.15, ' 0.11
       ROMEO:[salto]I   elige w          opciones: w 0.14, t 0.12, h 0.09
      ROMEO:[salto]I w  elige h          opciones: h 0.34, i 0.27, e 0.14
     ROMEO:[salto]I wh  elige a          opciones: a 0.41, o 0.22, e 0.20
     OMEO:[salto]I wha  elige t          opciones: t 0.79, s 0.05, l 0.04
     MEO:[salto]I what  elige [espacio]  opciones: [espacio] 0.76, , 0.06, h 0.06
     EO:[salto]I what   elige t          opciones: t 0.13, s 0.10, m 0.09
     O:[salto]I what t  elige h          opciones: h 0.70, o 0.19, i 0.04
     :[salto]I what th  elige e          opciones: e 0.38, a 0.28, o 0.13
     [salto]I what the  elige [espacio]  opciones: [espacio] 0.73, r 0.06, n 0.05
           I what the   elige t          opciones: t 0.11, s 0.10, m 0.08


### Diversidad por configuración

In [11]:
with open(os.path.join("data", "input.txt"), encoding="utf-8") as f:
    corpus_words = set(re.findall(r"[a-z']+", f.read().lower()))

print(f"{'config':8}{'entropía':>10}{'prob. top 5':>13}{'palabras':>10}{'distintas':>11}{'reales':>9}")
for name, (strategy, kwargs) in CONFIGS.items():
    tau = kwargs.get("tau", 1.0)
    probs = [F.softmax(lg[START:] / tau, dim=-1) for lg in logits_cache[name]]
    entropy = torch.cat([-(p * p.clamp_min(1e-12).log()).sum(-1) for p in probs]).mean().item()
    top5 = torch.cat([torch.topk(p, 5, dim=-1).values.sum(-1) for p in probs]).mean().item()
    words = [w for t in samples[name] for w in re.findall(r"[a-z']+", t[len(PROMPT):].lower())]
    real = 100 * sum(w in corpus_words for w in words) / len(words)
    print(f"{name:8}{entropy:>10.2f}{top5:>13.2f}{len(words):>10}{len(set(words)):>11}{real:>8.0f}%")

config    entropía  prob. top 5  palabras  distintas   reales
A             1.98         0.78       255          5      98%
B             1.50         0.86       217        103      77%
C             1.98         0.77       175        145      46%
D             1.51         0.85       235         79      90%
E             2.02         0.76       186        154      47%


### Incertidumbre en top-p (E)

In [12]:
def position_type(c):
    if c == "\n":
        return "después de salto de línea"
    if c == " ":
        return "después de espacio"
    if c in ",.;:!?'-":
        return "después de puntuación"
    return "dentro de palabra"


by_type = defaultdict(list)
for s, (text, logits) in enumerate(zip(samples["E"], logits_cache["E"]), 1):
    probs = F.softmax(logits / 1.0, dim=-1)
    nuc = nucleus_size(probs, 0.95)
    pmax = probs.max(-1).values
    steps = sorted(range(START, len(text) - 1), key=lambda j: (nuc[j].item(), -pmax[j].item()))
    print(f"Muestra {s} de E, núcleo promedio {nuc[START:].float().mean():.1f}")
    for label, js in (("seguro  ", steps[:2]), ("inseguro", steps[-2:])):
        for j in js:
            print(f"  {label} núcleo {nuc[j].item():2d}  prob. máxima {pmax[j]:.2f}  "
                  f"después de \"{show(text[max(0, j - 14) : j + 1])}\" eligió {show_char(text[j + 1])}")
    for j in range(START, len(text) - 1):
        by_type[position_type(text[j])].append(nuc[j].item())

print("\nTamaño promedio del núcleo (p = 0.95) según la posición:")
for k, v in sorted(by_type.items(), key=lambda kv: sum(kv[1]) / len(kv[1])):
    print(f"  {k:27} {sum(v) / len(v):5.1f}  (n={len(v)})")

Muestra 1 de E, núcleo promedio 12.9
  seguro   núcleo  1  prob. máxima 0.99  después de "ROMEO:" eligió [salto]
  seguro   núcleo  1  prob. máxima 0.98  después de "r toe![salto][salto]CLANET:" eligió [salto]
  inseguro núcleo 24  prob. máxima 0.15  después de "![salto][salto]CLANET:[salto]For " eligió h
  inseguro núcleo 28  prob. máxima 0.15  después de "whould benign: " eligió t
Muestra 2 de E, núcleo promedio 13.3
  seguro   núcleo  1  prob. máxima 0.99  después de "ROMEO:" eligió [salto]
  seguro   núcleo  1  prob. máxima 0.99  después de ".[salto][salto]MUKENTY REO:" eligió [salto]
  inseguro núcleo 30  prob. máxima 0.21  después de "hed I feem and[salto]" eligió E
  inseguro núcleo 32  prob. máxima 0.16  después de "imin.[salto][salto]MUKENTY " eligió R
Muestra 3 de E, núcleo promedio 12.7
  seguro   núcleo  1  prob. máxima 0.99  después de "ROMEO:" eligió [salto]
  seguro   núcleo  1  prob. máxima 0.99  después de "tar, now.[salto][salto]RIS:" eligió [salto]
  inseguro núcleo 

## Respuestas

a. Greedy (configuración A)

Greedy siempre elige el carácter con la probabilidad más alta. No usa nada de azar: si el texto anterior es el mismo, el siguiente carácter también será el mismo. Por eso las 5 muestras de A son idénticas.

El problema es que el modelo casi nunca está muy seguro. Después de "the " (con el espacio), la letra más probable es la t, pero solo con 0.12 de probabilidad. La s tiene 0.10 y la m 0.09. Greedy ignora todas las demás opciones, que juntas suman casi el 88 por ciento, y siempre escoge la t. Luego viene la h (0.49), después la e (0.33, casi empatada con la a, que tiene 0.32) y luego un espacio (0.72). Con eso vuelve a escribir "the " y queda en la misma situación que antes.

El modelo es pequeño y se fija sobre todo en los últimos caracteres. Cada vez que escribe "the " ve casi lo mismo y toma la misma decisión, así que queda atrapado en "the the the the" y no puede salir. En la tabla de diversidad, A tiene solo 5 palabras distintas de 255. Lo repetitivo no viene de que el modelo esté seguro, sino de que greedy siempre toma la opción más probable aunque gane por muy poco.

b. Top-k: B (k=5, tau=0.7) contra C (k=50, tau=1.0)

La temperatura divide los puntajes del modelo (logits) entre tau antes de convertirlos en probabilidades. Con tau menor que 1 las diferencias entre caracteres se hacen más grandes. Con tau igual a 0.7, la diferencia entre dos caracteres pesa como si fuera 1.43 veces mayor. Los caracteres que ya eran probables se vuelven todavía más probables y la probabilidad se concentra en pocas opciones. Con tau igual a 1.0 la distribución queda como la aprendió el modelo, más repartida.

En el análisis esto se ve en la entropía, que mide qué tan repartida está la probabilidad. Baja de 1.98 en C a 1.50 en B. La probabilidad que se llevan los 5 mejores caracteres sube de 0.77 en C a 0.86 en B.

El valor de k decide cuántos candidatos pueden salir. Con k igual a 5 solo se elige entre los 5 caracteres más probables, así que los caracteres raros nunca aparecen. Con k igual a 50, de un vocabulario de 65 caracteres, casi no se recorta nada, y pueden salir mayúsculas a media palabra o símbolos como el &. Subir k da más variedad, pero también más errores.

En las muestras se nota. B se lee como verso, por ejemplo "What make to him the to maress,", aunque repite palabras comunes, como en "make will will will me" o "the the a tange". Tiene 103 palabras distintas de 217, y el 77 por ciento existen en el texto de Shakespeare. C es mucho más variada, con 145 palabras distintas de 175, pero solo el 46 por ciento son palabras reales. Aparecen cosas como "Then upememen loody tecown theamight", "es&ashtlerss", "RGjo:" y personajes inventados como "HERCUCERY:".

La más coherente es B y la más creativa es C.

c. Top-p adaptativo (configuración E)

Top-p junta los caracteres más probables hasta sumar 0.95 de probabilidad y solo elige entre esos. A ese grupo se le llama núcleo. Si el modelo está seguro, el núcleo es pequeño. Si está dudando, el núcleo crece. El análisis mide el tamaño del núcleo en cada paso de las muestras de E.

Momentos en que el modelo está muy seguro (núcleo de 1 solo carácter):

1. Después del nombre de un personaje y los dos puntos, el modelo sabe que viene un salto de línea. Pasa con ROMEO: en las 5 muestras (probabilidad 0.99), con CLANET: en la muestra 1, MUKENTY REO: en la muestra 2, RIS: en la muestra 3 y EARILI: en la muestra 5.
2. Después de un punto al final de un verso también espera un salto de línea, como en "me, where feep." de la muestra 4 (probabilidad 0.97). En promedio, después de un signo de puntuación el núcleo es de solo 2.8 caracteres.

Momentos en que está muy inseguro (núcleo de 24 a 35 caracteres, y la opción más probable no pasa de 0.23):

1. Al empezar una palabra nueva, después de un espacio. Por ejemplo, después de "whould benign: " (núcleo de 28) o de "Goright kidin. " (núcleo de 35). Ahí puede empezar casi cualquier palabra. En promedio, el núcleo después de un espacio es de 21.7.
2. Al empezar una línea nueva. Por ejemplo, después de las líneas que terminan en "may alrear" (núcleo de 33), "trang tissul" (34) o "hone sto" (30). Puede venir otro verso o el nombre de otro personaje. En promedio, el núcleo después de un salto de línea es de 20.9.
3. Dentro de los nombres de personajes inventados, como MUKENTY en la muestra 2 o el nombre que empieza con QUEEN CI en la muestra 5 (núcleos de 32 y 34). En mayúsculas el modelo no sabe qué nombre está escribiendo.

En resumen, el modelo está seguro en la estructura del texto: después de los dos puntos de un nombre, después de la puntuación y en medio de palabras conocidas (núcleo promedio de 11.3). Duda donde empieza algo nuevo: una palabra, un verso o un nombre. Ahí top-p abre el núcleo a entre 20 y 35 opciones, y por eso en E aparecen palabras inventadas como "Bonoly!", "alrear" o "tissul".